# **Induction heads: Практика**

Induction head — голова внимания, которая на повторе «…[A][B] … [A]» смотрит на токен [B], шедший после прошлого [A], и копирует его в предсказание. [Olsson et al., 2022](https://arxiv.org/abs/2209.11895) определяют такие головы по поведению на повторенной случайной последовательности токенов; [Elhage et al., 2021](https://transformer-circuits.pub/2021/framework/index.html) разобрали их в двухслойной модели только с вниманием: голова предыдущего токена в первом слое и индукционная голова во втором, связанные K-композицией.

В этой практике вы возьмете такую модель — `attn-only-2l` из TransformerLens (2 слоя по 8 голов внимания, без MLP) — и найдете в ней обе головы тремя способами: по паттерну внимания, по абляции и по весам.

Задания пронумерованы, ответы на них вводятся в уроке «Induction heads: Практика».

## **Подготовка окружения**
Хватит CPU: модель маленькая.

In [ ]:
!pip install transformer-lens==3.9.0 -q

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)  # здесь только прямые проходы
model = HookedTransformer.from_pretrained("attn-only-2l", device="cpu")
print(model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_model)

## **Данные: повторенная случайная последовательность**

Возьмем 20 последовательностей из 50 случайных токенов и повторим каждую дважды, впереди поставим токен BOS. В первой половине следующий токен угадать нельзя: токены случайные. Во второй половине каждый токен уже встречался, и модель с induction heads может предсказать его, посмотрев, что шло после прошлого вхождения текущего токена.

Loss считаем на предсказании токенов 2–50 каждой половины: первый токен второй половины по-прежнему не следует ни из чего.

In [ ]:
L, B = 50, 20
gen = torch.Generator().manual_seed(0)
rand = torch.randint(1000, 10000, (B, L), generator=gen)
bos = torch.full((B, 1), model.tokenizer.bos_token_id)
tokens = torch.cat([bos, rand, rand], dim=1)  # [20, 1 + 2*50]


def half_losses(logits):
    """Средний loss предсказания токенов 2–50 первой и второй половины."""
    lp = utils.lm_cross_entropy_loss(logits, tokens, per_token=True)  # lp[:, j] — loss предсказания токена j + 1
    return ...  # Ваш код здесь


logits, cache = model.run_with_cache(tokens)
first, second = half_losses(logits)
print(f"loss: первая половина {first:.2f}, вторая половина {second:.2f}")

In [ ]:
lp = utils.lm_cross_entropy_loss(logits, tokens, per_token=True).mean(0)
plt.figure(figsize=(8, 3))
plt.plot(range(1, 2 * L + 1), lp)  # позиция 0 — BOS, первая половина — позиции 1…50
plt.axvline(L + 0.5, color="gray", linestyle="--")
plt.xlabel("позиция предсказываемого токена")
plt.ylabel("loss, среднее по 20 последовательностям")
plt.title("Loss на повторенной случайной последовательности")
plt.show()

**Задание 1. Чему равен средний loss на первой половине? Ответ округлите до сотых.**

**Задание 2. Чему равен средний loss на второй половине? Ответ округлите до сотых.**

## **Паттерн внимания: две оценки на голову**

Для каждой головы посчитаем две величины по второй половине, средние по позициям и последовательностям:

- **оценка предыдущего токена** — вес внимания с позиции $i$ на позицию $i-1$;
- **prefix-matching score** — вес внимания с позиции $i$ на позицию $i-L+1$: туда, где стоит токен, шедший после прошлого вхождения текущего токена.

Голова предыдущего токена дает высокую первую оценку, induction head — высокую вторую.

In [ ]:
prev_score = torch.zeros(2, model.cfg.n_heads)
prefix_score = torch.zeros(2, model.cfg.n_heads)
for layer in range(2):
    pattern = cache["pattern", layer]  # [батч, голова, позиция запроса, позиция ключа]
    # последние L элементов диагонали — запросы второй половины, позиции L+1…2L
    prev_score[layer] = ...  # Ваш код здесь
    prefix_score[layer] = ...  # Ваш код здесь

for layer in range(2):
    print(f"слой {layer}: prev", [round(x, 2) for x in prev_score[layer].tolist()])
    print(f"слой {layer}: prefix", [round(x, 2) for x in prefix_score[layer].tolist()])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (layer, head) in zip(axes, [(0, 3), (1, 6)]):
    ax.imshow(cache["pattern", layer][0, head], cmap="Blues")
    ax.set_title(f"голова {layer}.{head}, последовательность 0")
    ax.set_xlabel("позиция ключа")
    ax.set_ylabel("позиция запроса")
plt.show()

**Задание 3. У какой головы наибольший prefix-matching score?** Голову записывайте как «слой.голова», например 1.2.

**Задание 4. Чему равен этот prefix-matching score? Ответ округлите до сотых.**

**Задание 5. У какой головы слоя 0 наибольшая оценка предыдущего токена?**

**Задание 6. Чему равна эта оценка предыдущего токена? Ответ округлите до сотых.**

## **Абляция голов**

Паттерн внимания показывает, куда голова смотрит, но не то, нужна ли она для предсказания. Проверим вмешательством: обнулим выход одной головы (`hook_z`) и посмотрим, как изменится loss на второй половине. Это zero-абляция; в уроке «Абляции: zero/mean/resample и методология» разобрано, чем она отличается от mean- и resample-абляции.

In [ ]:
def ablated_losses(layer, head):
    """Loss первой и второй половины при обнуленном выходе одной головы."""
    def zero_head(z, hook):
        ...  # Ваш код здесь
        return z
    out = model.run_with_hooks(tokens, fwd_hooks=[(utils.get_act_name("z", layer), zero_head)])
    return half_losses(out)


abl = torch.zeros(2, model.cfg.n_heads)
for layer in range(2):
    for head in range(model.cfg.n_heads):
        abl[layer, head] = ablated_losses(layer, head)[1]
for layer in range(2):
    print(f"слой {layer}:", [round(x, 2) for x in abl[layer].tolist()])

**Задание 7. Чему равен loss на второй половине после абляции головы из задания 3? Ответ округлите до сотых.**

**Задание 8. Абляция какой головы слоя 0 сильнее всего поднимает loss на второй половине?**

## **Композиция по весам**

Elhage et al. меряют, насколько голова второго слоя читает выход головы первого слоя, нормой Фробениуса произведения матриц: для K-композиции $\|W_{QK}^{h_2}W_{OV}^{h_1}\|_F / (\|W_{QK}^{h_2}\|_F\,\|W_{OV}^{h_1}\|_F)$, для Q- и V-композиции — аналогично с $W_{QK}^{h_2\,T}$ и $W_{OV}^{h_2}$. В TransformerLens это `model.all_composition_scores(mode)`. Данные для этой оценки не нужны: она считается по весам. В статье Elhage et al. из этой оценки вычитают ее среднее для случайных матриц той же формы; `all_composition_scores` возвращает оценку без этой поправки, поэтому ее сравнивают с другими оценками той же модели.

In [ ]:
h1, h2 = (0, 3), (1, 6)  # голова предыдущего токена и induction head
for mode in "QKV":
    scores = model.all_composition_scores(mode)  # [layer1, head1, layer2, head2]
    ...  # Ваш код здесь

**Задание 9. Какой тип композиции между головами 0.3 и 1.6 дает наибольшую оценку?**

## **Заключение**

- На повторенной случайной последовательности loss второй половины в несколько раз ниже, чем первой: модель копирует продолжение из контекста.
- Паттерн внимания выделяет две головы: голову предыдущего токена в слое 0 и induction head в слое 1 с prefix-matching score около 0.7.
- Абляция подтверждает, что обе нужны для копирования: без любой из них loss на второй половине поднимается в несколько раз, а голова слоя 0 с наибольшей оценкой предыдущего токена — та же, абляция которой сильнее всех поднимает loss.
- Веса показывают, через какой канал связаны две головы: оценка K-композиции между ними выше Q- и V-композиции, как и предсказывает схема Elhage et al.